# Python Project - Crime Data Pipeline and Reporting Dataset
**Author: Fenner Backhouse**  
This notebook builds a repeatable data engineering pipeline for UK Police street-level crime data. It follows the pipeline layers listed in the project brief:

1. Ingestion layer
2. Cleaning and validation layer
3. Feature engineering and transformation layer
4. Aggregation for reporting layer
5. Export layer

**Selected police force areas**

- Cleveland
- West Yorkshire
- North Yorkshire
- Wiltshire

**Final reporting grain**

`police_force_name × year_month × crime_type`

The final output is a BI-ready CSV containing aggregated crime counts, deprivation context, population values, and normalised crime rates.

## 0. Project Setup

Setup section to define imports, file paths, selected police forces, date ranges, and reusable configuration values that will be used throughout the pipeline.

In [7]:
import pandas as pd
import numpy as np

In [9]:
#### Project file paths ####
# Folder containing monthly Police.uk CSV folders
crimedata_path = "Data/CL_WY_NY_WI_Apr2023-Mar2026_Crime/"

# Enrichment datasets
population_path = "Data/policeforceareas1991to2024.xlsx"
socioeconomic_path = "Data/File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv"

# Output file
output_path = "Data/AggregateCrimeDatasetReport.csv"


#### Selected police forces ####
# These names match the Police.uk CSV filenames
police_forces = [
    "cleveland",
    "west-yorkshire",
    "north-yorkshire",
    "wiltshire"
]

# These names match the ONS population file and are also used in the final reporting dataset.
force_name_map = {
    "cleveland": "Cleveland",
    "west-yorkshire": "West Yorkshire",
    "north-yorkshire": "North Yorkshire",
    "wiltshire": "Wiltshire"
}

selected_force_names = list(force_name_map.values())

# Date range: April 2023 to March 2026
date_range = pd.date_range(
    start="2023-04",
    end="2026-03",
    freq="MS"
).strftime("%Y-%m")

print("Selected forces:", selected_force_names)
print("Number of months expected:", len(date_range))
print("First month:", date_range[0])
print("Last month:", date_range[-1])

Selected forces: ['Cleveland', 'West Yorkshire', 'North Yorkshire', 'Wiltshire']
Number of months expected: 36
First month: 2023-04
Last month: 2026-03


# 1. Ingestion Layer

The ingestion layer reads raw source datasets into Python with minimal alteration.

For the UK Police crime data, the files are read iteratively in batches by police force and months without loading the full dataset into memory.

## 1.1 Ingest UK Police Crime Data

This section reads the monthly Police.uk street-level crime CSV files for the selected police forces. The loop stores one merged dataframe per police force in a dictionary called `force_dfs` and logs loaded or missing files.

In [16]:
# Initialise empty dictionary and dfs to append into
force_dfs = {}
missing_files = []
ingestion_log = []

# Loop through each force one at a time
for force in police_forces:
    force_data = []

    # Loop through each month in the date range one at a time
    for date in date_range:
        readin_path = (crimedata_path + date + "/" + date + "-" + force + "-street.csv")
        
        try:
            month_data = pd.read_csv(readin_path)

            # Add source tracking columns so the source remains clear after concatenation.
            month_data["source_file_month"] = date
            month_data["force_file_name"] = force

            force_data.append(month_data)

            ingestion_log.append({
                "force_file_name": force,
                "source_file_month": date,
                "file_path": readin_path,
                "rows_read": len(month_data),
                "status": "loaded"
            })

        # Error tracking for file path issues
        except FileNotFoundError:
            missing_files.append(readin_path)
            ingestion_log.append({
                "force_file_name": force,
                "source_file_month": date,
                "file_path": readin_path,
                "rows_read": 0,
                "status": "missing"
            })
            continue

    if len(force_data) > 0:
        force_dfs[force] = pd.concat(force_data, ignore_index=True)
    else:
        force_dfs[force] = pd.DataFrame()

crime_ingestion_log = pd.DataFrame(ingestion_log)

# Stop the pipeline if no crime data has been loaded.
if sum(len(df) for df in force_dfs.values()) == 0:
    raise ValueError("No crime data was loaded. Check crimedata_path, date folders, and file naming pattern.")

print("Number of force dataframes loaded:", len(force_dfs))
print("Total missing files:", len(missing_files))

crime_ingestion_log.head()

Number of force dataframes loaded: 4
Total missing files: 0


,force_file_name,source_file_month,file_path,rows_read,status
0,cleveland,2023-04,Data/CL_WY_NY_WI_Apr2023-Mar2026_Crime/2023-04...,8707,loaded
1,cleveland,2023-05,Data/CL_WY_NY_WI_Apr2023-Mar2026_Crime/2023-05...,8764,loaded
2,cleveland,2023-06,Data/CL_WY_NY_WI_Apr2023-Mar2026_Crime/2023-06...,8719,loaded
3,cleveland,2023-07,Data/CL_WY_NY_WI_Apr2023-Mar2026_Crime/2023-07...,8556,loaded
4,cleveland,2023-08,Data/CL_WY_NY_WI_Apr2023-Mar2026_Crime/2023-08...,8299,loaded


In [20]:
# Review any missing files
crime_ingestion_log[crime_ingestion_log["status"] == "missing"].head()

,force_file_name,source_file_month,file_path,rows_read,status


## 1.2 Combine Ingested Crime Files into One Raw Staging Dataframe

Individual force-level dataframes are combined into a single raw dataframe before cleaning and transformation.

In [24]:
crime_raw = pd.concat(force_dfs.values(), ignore_index=True)

print("Combined raw crime rows:", len(crime_raw))
print("Combined raw crime columns:", crime_raw.shape[1])

crime_raw.head()

Combined raw crime rows: 1541355
Combined raw crime columns: 14


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context,source_file_month,force_file_name
0,NaN,2023-04,Cleveland Police,Cleveland Police,-1.235660,54.710526,On or near Dobson Place,E01011954,Hartlepool 001A,Anti-social behaviour,NaN,NaN,2023-04,cleveland
1,d6de0a0661244386645bc3435c8319194fd27234272d29...,2023-04,Cleveland Police,Cleveland Police,-1.233941,54.711980,On or near Wolsingham Road,E01011954,Hartlepool 001A,Criminal damage and arson,Investigation complete; no suspect identified,NaN,2023-04,cleveland
2,7ea9cd133d761ac9c85a4367bab92d7e390b475949936e...,2023-04,Cleveland Police,Cleveland Police,-1.239173,54.711061,On or near Butterwick Road,E01011954,Hartlepool 001A,Criminal damage and arson,Investigation complete; no suspect identified,NaN,2023-04,cleveland
3,0913b9503bc8873b9134c84b61410c5d1a4e72069a1e35...,2023-04,Cleveland Police,Cleveland Police,-1.239498,54.711980,On or near King Oswy Drive,E01011954,Hartlepool 001A,Public order,Investigation complete; no suspect identified,NaN,2023-04,cleveland
4,ecb46016e7cf86b64caba450a6b913ab1e4778cc3dee5f...,2023-04,Cleveland Police,Cleveland Police,-1.239865,54.710589,On or near Marshall Close,E01011954,Hartlepool 001A,Public order,Unable to prosecute suspect,NaN,2023-04,cleveland


In [30]:
# Ingestion completeness check: rows loaded by force and month
rows_by_force_month = (
    crime_raw
    .groupby(["force_file_name", "source_file_month"])
    .size()
    .reset_index(name="row_count")
)

# Sort from lowest row count to spot anomalously low counts
rows_by_force_month.sort_values(by='row_count', ascending=True).head()

,force_file_name,source_file_month,row_count
118,wiltshire,2024-02,3744
130,wiltshire,2025-02,3757
129,wiltshire,2025-01,3830
128,wiltshire,2024-12,3835
117,wiltshire,2024-01,3842


## 1.3 Ingest Population Enrichment Data

This section reads the ONS police force area population workbook. The relevant sheet is `Mid-2021 to Mid-2024` because it contains the most recent available population estimates in the file.

Only the file read happens here. Column selection, total population calculation, temporal handling, and joining are done later in the transformation layer.

In [33]:
population_raw = pd.read_excel(
    population_path,
    sheet_name="Mid-2021 to Mid-2024",
    header=3
)

print("Population raw shape:", population_raw.shape)
population_raw.head()

Population raw shape: (172, 175)


,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M76,M77,M78,M79,M80,M81,M82,M83,M84,M85
0,E23000001,Metropolitan Police,2021,51645,52111,51165,51033,51843,52101,51137,...,18075,17677,15864,14156,12511,13118,12214,11284,10136,51460
1,E23000001,Metropolitan Police,2022,53387,51282,51393,50706,50616,51641,52319,...,18979,17292,16873,15113,13373,11745,12245,11321,10450,52766
2,E23000001,Metropolitan Police,2023,51047,53405,51051,51359,50792,50682,52018,...,23619,18192,16512,15945,14302,12562,10881,11329,10369,53824
3,E23000001,Metropolitan Police,2024,51526,51041,52918,50777,51276,50664,50846,...,21814,22697,17361,15658,15144,13481,11793,10164,10433,55460
4,E23000002,Cumbria,2021,2069,2089,2198,2182,2384,2432,2374,...,2413,2379,2199,1913,1733,1660,1509,1392,1257,5864


## 1.4 Ingest Socioeconomic / Deprivation Enrichment Data

This section reads the Index of Multiple Deprivation dataset. No cleaning is applied yet. The intended join key is `LSOA code`, because Police.uk crime records include LSOA information.

In [36]:
socioeconomic_raw = pd.read_csv(socioeconomic_path)

print("Socioeconomic raw shape:", socioeconomic_raw.shape)
socioeconomic_raw.head()

Socioeconomic raw shape: (33755, 56)


,LSOA code (2021),LSOA name (2021),Local Authority District code (2024),Local Authority District name (2024),Index of Multiple Deprivation (IMD) Score,Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived),Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs),Income Score (rate),Income Rank (where 1 is most deprived),Income Decile (where 1 is most deprived 10% of LSOAs),...,Indoors Sub-domain Score,Indoors Sub-domain Rank (where 1 is most deprived),Indoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Outdoors Sub-domain Score,Outdoors Sub-domain Rank (where 1 is most deprived),Outdoors Sub-domain Decile (where 1 is most deprived 10% of LSOAs),Total population: mid 2022,Dependent Children aged 0-15: mid 2022,Older population aged 60 and over: mid 2022,Working age population 18-66 (for use with Employment Deprivation Domain): mid 2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,...,1.207,1105,1,1.414,1586,1,1795,149,520,1248
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,...,0.355,9591,3,1.839,592,1,1671,81,387,1324
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8,...,0.318,10175,4,1.679,903,1,1896,136,432,1469
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,...,0.012,15502,5,2.065,303,1,1737,177,160,1448
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,...,0.399,8934,3,0.400,9136,3,1837,397,225,1260


In [38]:
# Inspect column names to confirm the available deprivation fields.
socioeconomic_raw.columns.tolist()

['LSOA code (2021)',
 'LSOA name (2021)',
 'Local Authority District code (2024)',
 'Local Authority District name (2024)',
 'Index of Multiple Deprivation (IMD) Score',
 'Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)',
 'Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)',
 'Income Score (rate)',
 'Income Rank (where 1 is most deprived)',
 'Income Decile (where 1 is most deprived 10% of LSOAs)',
 'Employment Score (rate)',
 'Employment Rank (where 1 is most deprived)',
 'Employment Decile (where 1 is most deprived 10% of LSOAs)',
 'Education, Skills and Training Score',
 'Education, Skills and Training Rank (where 1 is most deprived)',
 'Education, Skills and Training Decile (where 1 is most deprived 10% of LSOAs)',
 'Health Deprivation and Disability Score',
 'Health Deprivation and Disability Rank (where 1 is most deprived)',
 'Health Deprivation and Disability Decile (where 1 is most deprived 10% of LSOAs)',
 'Crime Score',
 'Cr

# 2. Cleaning and Validation Layer

The cleaning and validation layer standardises source columns, handles missing values and duplicates where appropriate, checks required fields, and prepares each dataset for later transformation or joining.

The duplicate check at the final reporting grain is carried out after aggregation, because the reporting grain does not exist until the aggregation layer.

## 2.1 Clean and Validate Crime Data

This section standardises Police.uk column names, maps file-based force names to official police force names, removes exact duplicate records, and checks source fields needed for the final reporting grain.

Date-derived columns such as `year`, `month_number`, and `season` are created later in the feature engineering layer.

In [42]:
crime_clean = crime_raw.copy()

rows_before_cleaning = len(crime_clean)

# Standardise column names from Police.uk format to analysis-friendly names
crime_clean = crime_clean.rename(columns={
    "Crime ID": "crime_id",
    "Month": "month",
    "Reported by": "reported_by",
    "Falls within": "falls_within",
    "Longitude": "longitude",
    "Latitude": "latitude",
    "Location": "location",
    "LSOA code": "lsoa_code",
    "LSOA name": "lsoa_name",
    "Crime type": "crime_type",
    "Last outcome category": "last_outcome_category",
    "Context": "context"
})

# Create official police force name using the force filename value
crime_clean["police_force_name"] = crime_clean["force_file_name"].map(force_name_map)

# Remove exact duplicate rows only
# Do not drop rows just because crime_id is missing as some valid Police.uk records have missing crime_id
exact_duplicate_rows = crime_clean.duplicated().sum()
crime_clean = crime_clean.drop_duplicates().copy()

# Remove context as entirely null and not useful for aggregate analysis
if "context" in crime_clean.columns:
    crime_clean = crime_clean.drop(columns=["context"])

print("Rows before crime cleaning:", rows_before_cleaning)
print("Exact duplicate rows removed:", exact_duplicate_rows)
print("Rows after crime cleaning:", len(crime_clean))

crime_clean.head()

Rows before crime cleaning: 1541355
Exact duplicate rows removed: 58625
Rows after crime cleaning: 1482730


,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,source_file_month,force_file_name,police_force_name
0,NaN,2023-04,Cleveland Police,Cleveland Police,-1.235660,54.710526,On or near Dobson Place,E01011954,Hartlepool 001A,Anti-social behaviour,NaN,2023-04,cleveland,Cleveland
1,d6de0a0661244386645bc3435c8319194fd27234272d29...,2023-04,Cleveland Police,Cleveland Police,-1.233941,54.711980,On or near Wolsingham Road,E01011954,Hartlepool 001A,Criminal damage and arson,Investigation complete; no suspect identified,2023-04,cleveland,Cleveland
2,7ea9cd133d761ac9c85a4367bab92d7e390b475949936e...,2023-04,Cleveland Police,Cleveland Police,-1.239173,54.711061,On or near Butterwick Road,E01011954,Hartlepool 001A,Criminal damage and arson,Investigation complete; no suspect identified,2023-04,cleveland,Cleveland
3,0913b9503bc8873b9134c84b61410c5d1a4e72069a1e35...,2023-04,Cleveland Police,Cleveland Police,-1.239498,54.711980,On or near King Oswy Drive,E01011954,Hartlepool 001A,Public order,Investigation complete; no suspect identified,2023-04,cleveland,Cleveland
4,ecb46016e7cf86b64caba450a6b913ab1e4778cc3dee5f...,2023-04,Cleveland Police,Cleveland Police,-1.239865,54.710589,On or near Marshall Close,E01011954,Hartlepool 001A,Public order,Unable to prosecute suspect,2023-04,cleveland,Cleveland


In [50]:
# Check how many rows are missing crime_id
missing_crime_id_count = crime_clean["crime_id"].isna().sum()
missing_crime_id_pct = crime_clean["crime_id"].isna().mean() * 100

print("Rows missing crime_id:", missing_crime_id_count)
print("Percentage missing crime_id:", round(missing_crime_id_pct, 2), "%")

# Check how many of these are recorded as Anti-social behaviour (often recorded without crime_id and therefore likely valid), justifying retention
crime_clean.loc[
    crime_clean["crime_id"].isna(),
    "crime_type"
].value_counts()

# Confirmation of 'context' Null count in raw crime data, justifying removal in previous cleaning
crime_raw.info()

Rows missing crime_id: 153030
Percentage missing crime_id: 10.32 %
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1541355 entries, 0 to 1541354
Data columns (total 14 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Crime ID               1330042 non-null  object 
 1   Month                  1541355 non-null  object 
 2   Reported by            1541355 non-null  object 
 3   Falls within           1541355 non-null  object 
 4   Longitude              1530393 non-null  float64
 5   Latitude               1530393 non-null  float64
 6   Location               1541355 non-null  object 
 7   LSOA code              1530391 non-null  object 
 8   LSOA name              1530391 non-null  object 
 9   Crime type             1541355 non-null  object 
 10  Last outcome category  1330042 non-null  object 
 11  Context                0 non-null        float64
 12  source_file_month      1541355 non-null  object 
 13  force

In [52]:
# Required source fields needed before feature engineering and aggregation.
required_crime_source_cols = [
    "police_force_name",
    "month",
    "crime_type"
]

print("Missing values in required crime source fields:")
print(crime_clean[required_crime_source_cols].isna().sum())

Missing values in required crime source fields:
police_force_name    0
month                0
crime_type           0
dtype: int64


In [54]:
# Drop records that cannot be placed into the final reporting grain.
rows_before_required_drop = len(crime_clean)
crime_clean = crime_clean.dropna(subset=required_crime_source_cols).copy()
rows_after_required_drop = len(crime_clean)

print("Rows before required-field drop:", rows_before_required_drop)
print("Rows after required-field drop:", rows_after_required_drop)
print("Rows removed due to missing required source fields:", rows_before_required_drop - rows_after_required_drop)

Rows before required-field drop: 1482730
Rows after required-field drop: 1482730
Rows removed due to missing required source fields: 0


In [56]:
# Validate expected force coverage and crime categories after cleaning.
print("Unique police forces:")
print(crime_clean["police_force_name"].unique())

print("Crime type counts:")
print(crime_clean["crime_type"].value_counts().sort_index())

Unique police forces:
['Cleveland' 'West Yorkshire' 'North Yorkshire' 'Wiltshire']
Crime type counts:
crime_type
Anti-social behaviour           153030
Bicycle theft                    11291
Burglary                         68756
Criminal damage and arson       123375
Drugs                            46037
Other crime                      41531
Other theft                      93287
Possession of weapons            12656
Public order                    123822
Robbery                          15035
Shoplifting                     118494
Theft from the person            10543
Vehicle crime                    73586
Violence and sexual offences    591287
Name: count, dtype: int64


## 2.2 Clean and Validate Socioeconomic / IMD Data

This section selects the deprivation fields needed for enrichment and validates that there is only one deprivation row per LSOA code. This prevents unintended many-to-many joins later.

In [59]:
# Clean and select socioeconomic deprivation data
socioeconomic_df = socioeconomic_raw.copy()

# Renames socioeconomic dataset columns for ease of use
socioeconomic_clean = socioeconomic_df.rename(columns={
    "LSOA code (2021)": "lsoa_code",
    "LSOA name (2021)": "lsoa_name_2021",
    "Local Authority District code (2024)": "local_authority_district_code_2024",
    "Local Authority District name (2024)": "local_authority_district_name_2024",
    "Index of Multiple Deprivation (IMD) Score": "imd_score",
    "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)": "imd_rank",
    "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)": "imd_decile",
    "Income Score (rate)": "income_score",
    "Income Rank (where 1 is most deprived)": "income_rank",
    "Income Decile (where 1 is most deprived 10% of LSOAs)": "income_decile",
    "Employment Score (rate)": "employment_score",
    "Employment Rank (where 1 is most deprived)": "employment_rank",
    "Employment Decile (where 1 is most deprived 10% of LSOAs)": "employment_decile",
    "Crime Score": "imd_crime_score",
    "Crime Rank (where 1 is most deprived)": "imd_crime_rank",
    "Crime Decile (where 1 is most deprived 10% of LSOAs)": "imd_crime_decile",
    "Total population: mid 2022": "lsoa_population_mid_2022"
})

socioeconomic_clean = socioeconomic_clean[
    [
        "lsoa_code",
        "lsoa_name_2021",
        "local_authority_district_code_2024",
        "local_authority_district_name_2024",
        "imd_score",
        "imd_rank",
        "imd_decile",
        "income_score",
        "income_rank",
        "income_decile",
        "employment_score",
        "employment_rank",
        "employment_decile",
        "imd_crime_score",
        "imd_crime_rank",
        "imd_crime_decile",
        "lsoa_population_mid_2022"
    ]
].copy()

socioeconomic_clean.head()

,lsoa_code,lsoa_name_2021,local_authority_district_code_2024,local_authority_district_name_2024,imd_score,imd_rank,imd_decile,income_score,income_rank,income_decile,employment_score,employment_rank,employment_decile,imd_crime_score,imd_crime_rank,imd_crime_decile,lsoa_population_mid_2022
0,E01000001,City of London 001A,E09000001,City of London,8.742,26525,8,0.013,33730,10,0.014,33708,10,-2.220,33698,10,1795
1,E01000002,City of London 001B,E09000001,City of London,4.722,31203,10,0.018,33669,10,0.010,33734,10,-2.277,33712,10,1671
2,E01000003,City of London 001C,E09000001,City of London,9.250,25913,8,0.107,25167,8,0.064,26985,8,-0.765,27325,9,1896
3,E01000005,City of London 001E,E09000001,City of London,19.884,14807,5,0.211,14836,5,0.104,17911,6,-0.626,25630,8,1737
4,E01000006,Barking and Dagenham 016A,E09000002,Barking and Dagenham,25.307,10917,4,0.343,7519,3,0.120,15286,5,-0.072,17875,6,1837


In [61]:
# Socioeconomic deprivation validation checks
print("Socioeconomic rows before duplicate handling:", len(socioeconomic_clean))
print(
    "Duplicate LSOA codes before duplicate handling:",
    socioeconomic_clean["lsoa_code"].duplicated().sum()
)

print("Missing values before duplicate handling:")
print(socioeconomic_clean.isna().sum())

# Keep one row per LSOA if any duplicates exist
socioeconomic_clean = socioeconomic_clean.drop_duplicates(
    subset=["lsoa_code"]
).copy()

print("Socioeconomic rows after duplicate handling:", len(socioeconomic_clean))
print(
    "Duplicate LSOA codes after duplicate handling:",
    socioeconomic_clean["lsoa_code"].duplicated().sum()
)

assert socioeconomic_clean["lsoa_code"].duplicated().sum() == 0

Socioeconomic rows before duplicate handling: 33755
Duplicate LSOA codes before duplicate handling: 0
Missing values before duplicate handling:
lsoa_code                             0
lsoa_name_2021                        0
local_authority_district_code_2024    0
local_authority_district_name_2024    0
imd_score                             0
imd_rank                              0
imd_decile                            0
income_score                          0
income_rank                           0
income_decile                         0
employment_score                      0
employment_rank                       0
employment_decile                     0
imd_crime_score                       0
imd_crime_rank                        0
imd_crime_decile                      0
lsoa_population_mid_2022              0
dtype: int64
Socioeconomic rows after duplicate handling: 33755
Duplicate LSOA codes after duplicate handling: 0


## 2.3 Clean and Validate Population Data

This section selects the population fields required for reporting, calculates total population from age-sex columns, filters to the selected police force areas, and validates that the population data has one row per police force and year before it is used in a join.

In [64]:
population_df = population_raw.copy()

# Identify age-sex columns such as F0, F1, M0, M1 etc.
age_sex_cols = [
    col for col in population_df.columns
    if str(col).startswith("F") or str(col).startswith("M")
]

population_df["population_total"] = population_df[age_sex_cols].sum(axis=1)

population_clean = population_df[
    ["PFA 2023 Code", "PFA 2023 Name", "Year", "population_total"]
].copy()

population_clean = population_clean.rename(columns={
    "PFA 2023 Code": "police_force_code",
    "PFA 2023 Name": "police_force_name",
    "Year": "year"
})

# Filter to the selected four forces using official names
population_clean = population_clean[
    population_clean["police_force_name"].isin(selected_force_names)
].copy()

print("Population rows after force filter:", len(population_clean))
print(population_clean[["police_force_name", "year", "population_total"]])

Population rows after force filter: 16
    police_force_name  year  population_total
32    North Yorkshire  2021            820478
33    North Yorkshire  2022            828803
34    North Yorkshire  2023            835592
35    North Yorkshire  2024            844571
36     West Yorkshire  2021           2350407
37     West Yorkshire  2022           2376124
38     West Yorkshire  2023           2408679
39     West Yorkshire  2024           2435236
48          Cleveland  2021            570155
49          Cleveland  2022            581486
50          Cleveland  2023            591464
51          Cleveland  2024            600369
148         Wiltshire  2021            747000
149         Wiltshire  2022            752231
150         Wiltshire  2023            759997
151         Wiltshire  2024            767575


In [66]:
# Population cleaning validation checks
print("Missing population values before temporal extension:")
print(population_clean.isna().sum())

population_rows_per_key_clean = population_clean.groupby(["police_force_name", "year"]).size()

print("Rows per force and year before temporal extension. Each should be 1:")
print(population_rows_per_key_clean)

assert population_rows_per_key_clean.max() == 1
assert population_clean["population_total"].isna().sum() == 0

Missing population values before temporal extension:
police_force_code    0
police_force_name    0
year                 0
population_total     0
dtype: int64
Rows per force and year before temporal extension. Each should be 1:
police_force_name  year
Cleveland          2021    1
                   2022    1
                   2023    1
                   2024    1
North Yorkshire    2021    1
                   2022    1
                   2023    1
                   2024    1
West Yorkshire     2021    1
                   2022    1
                   2023    1
                   2024    1
Wiltshire          2021    1
                   2022    1
                   2023    1
                   2024    1
dtype: int64
